# 第4回：近傍・1段の決定木・多数決

学習目標：近い順と投票を追い、学習と推論、検証での選択と最終テストを区別する。

このNotebookは必要なコードを収録し、同梱Python・JSONをimport／読み込みしない。Python 3.10以降の記法・標準ライブラリのみ。カーネル再起動後に先頭から全セルを実行する。環境は`../programs/README.md`のvenvまたはuv環境を使う。

架空の固定データ：学習7件・検証4件・テスト4件。外部データなし、乱数なし。特徴x・yは整数、ラベルはA／B。IDは距離に含めない。同距離はID順、同票はA、同検証成績なら小さいk。距離の二乗とGiniは分数で厳密に計算し、基準照合の浮動小数点許容誤差は0（浮動小数点を使わない）。RF・SVMの実装ではない。

In [ ]:
import sys
print(sys.executable)
print(sys.version)

## 処理コード

データ、近傍探索、投票、1段決定木、評価の順。`predict`に入力の正解を渡さず、`evaluate`で予測後に答え合わせする。

In [ ]:
"""第4回：k近傍法・1段の決定木・多数決を固定した架空データで観察する。

Python標準ライブラリのみ。距離の二乗・Gini不純度はFractionで厳密に計算。
ランダムフォレストやSVMを実装したプログラムではありません。
"""
import argparse
from collections import Counter
from fractions import Fraction
import json


# ID、特徴x、特徴y、正解ラベル。実物の測定値ではない。
TRAIN = (("T1", 0, 0, "A"), ("T2", 0, 2, "A"),
         ("T3", 2, 0, "A"), ("T4", 2, 2, "B"),
         ("T5", 6, 6, "B"), ("T6", 6, 8, "B"), ("T7", 8, 6, "B"))
VALIDATION = (("V1", 2, 3, "A"), ("V2", 3, 2, "A"),
              ("V3", 7, 7, "B"), ("V4", 8, 8, "B"))
TEST = (("E1", 1, 1, "A"), ("E2", 3, 3, "A"),
        ("E3", 7, 6, "B"), ("E4", 6, 7, "B"))
QUERY = (2, 3)


def validate_rows(rows):
    if not rows:
        raise ValueError("データは1件以上必要です")
    ids = []
    for row in rows:
        if len(row) != 4:
            raise ValueError("行はID・x・y・ラベルの4項目です")
        identity, x, y, label = row
        if not isinstance(identity, str) or not identity:
            raise ValueError("IDは空でない文字列です")
        if type(x) is not int or type(y) is not int or label not in ("A", "B"):
            raise ValueError("特徴は整数、ラベルはAまたはBです")
        ids.append(identity)
    if len(ids) != len(set(ids)):
        raise ValueError("IDが重複しています")


def validate_point(point):
    if len(point) != 2 or any(type(value) is not int for value in point):
        raise ValueError("入力は整数の特徴2個です")


def majority(labels):
    if not labels or any(label not in ("A", "B") for label in labels):
        raise ValueError("投票は1票以上、AまたはBです")
    counts = Counter(labels)
    # 同票ならA。この教材の規約であり、普遍的な規則ではない。
    return min(counts, key=lambda label: (-counts[label], label))


def nearest(train, point, k, scales=(1, 1)):
    validate_rows(train)
    validate_point(point)
    if type(k) is not int or not 1 <= k <= len(train):
        raise ValueError("kは1以上、学習件数以下の整数です")
    if len(scales) != 2 or any(type(s) is not int or s <= 0 for s in scales):
        raise ValueError("尺度は正の整数2個です")
    ranked = []
    for identity, x, y, label in train:
        d2 = Fraction((x - point[0]) ** 2, scales[0] ** 2)
        d2 += Fraction((y - point[1]) ** 2, scales[1] ** 2)
        ranked.append((d2, identity, label))
    # 距離が等しい場合はID順。元のTRAINを書き換えない。
    return sorted(ranked)[:k]


def predict_knn(train, point, k, scales=(1, 1)):
    return majority([label for _, _, label in nearest(train, point, k, scales)])


def gini(rows):
    validate_rows(rows)
    counts = Counter(row[3] for row in rows)
    return 1 - sum(Fraction(n, len(rows)) ** 2 for n in counts.values())


def fit_stump(train):
    """候補の加重Giniが最小となる1回の分岐を学習側だけで選ぶ。"""
    validate_rows(train)
    base = majority([row[3] for row in train])
    candidates = []
    for axis in (0, 1):
        values = sorted({row[axis + 1] for row in train})
        for low, high in zip(values, values[1:]):
            threshold = Fraction(low + high, 2)
            left = [row for row in train if row[axis + 1] <= threshold]
            right = [row for row in train if row[axis + 1] > threshold]
            weighted = (len(left) * gini(left) + len(right) * gini(right)) / len(train)
            candidates.append((weighted, axis, threshold, left, right))
    if not candidates:
        return {"axis": None, "threshold": None, "left": base, "right": base,
                "weighted_gini": gini(train)}
    # 同じ不純度ならx優先、次に小さいしきい値。改善しなければ分岐しない。
    weighted, axis, threshold, left, right = min(candidates, key=lambda c: c[:3])
    if weighted >= gini(train):
        return {"axis": None, "threshold": None, "left": base, "right": base,
                "weighted_gini": gini(train)}
    return {"axis": axis, "threshold": threshold,
            "left": majority([r[3] for r in left]),
            "right": majority([r[3] for r in right]), "weighted_gini": weighted}


def predict_stump(model, point):
    validate_point(point)
    if model["axis"] is None or point[model["axis"]] <= model["threshold"]:
        return model["left"]
    return model["right"]


def evaluate(predictor, rows):
    validate_rows(rows)
    predictions = [predictor((r[1], r[2])) for r in rows]
    return {"predictions": predictions,
            "correct": sum(pred == row[3] for pred, row in zip(predictions, rows)),
            "total": len(rows)}


def choose_k(validation_scores):
    if not validation_scores:
        raise ValueError("検証結果が必要です")
    # 比較する候補は同じ検証データを使う。同成績なら小さいk。
    return min(validation_scores, key=lambda k: (
        -Fraction(validation_scores[k]["correct"], validation_scores[k]["total"]), k))


def run_experiment():
    for rows in (TRAIN, VALIDATION, TEST):
        validate_rows(rows)
    ids = [row[0] for rows in (TRAIN, VALIDATION, TEST) for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("分割間でIDが重複しています")
    train_scores = {k: evaluate(lambda point: predict_knn(TRAIN, point, k), TRAIN)
                    for k in (1, 3)}
    validation_scores = {k: evaluate(lambda point: predict_knn(TRAIN, point, k), VALIDATION)
                         for k in (1, 3)}
    selected = choose_k(validation_scores)
    # テストはkを確定した後、選んだモデルだけに使う。
    test_score = evaluate(lambda point: predict_knn(TRAIN, point, selected), TEST)
    stump = fit_stump(TRAIN)
    votes = [predict_knn(TRAIN, QUERY, 1), predict_knn(TRAIN, QUERY, 3),
             predict_stump(stump, QUERY)]
    scale_rows = (("S1", 0, 3, "A"), ("S2", 2, 0, "B"))
    return {
        "data": {"train": [list(r) for r in TRAIN],
                 "validation": [list(r) for r in VALIDATION], "test": [list(r) for r in TEST]},
        "query": list(QUERY),
        "nearest3": [{"id": i, "distance_squared": str(d), "label": label}
                     for d, i, label in nearest(TRAIN, QUERY, 3)],
        "train_scores": {str(k): score for k, score in train_scores.items()},
        "validation_scores": {str(k): score for k, score in validation_scores.items()},
        "selected_k": selected, "selected_test": test_score,
        "stump": {**stump, "threshold": str(stump["threshold"]),
                  "weighted_gini": str(stump["weighted_gini"]),
                  "query_prediction": predict_stump(stump, QUERY)},
        "votes": votes, "vote_prediction": majority(votes),
        "scale_example": {"raw": predict_knn(scale_rows, (0, 0), 1),
                          "y_divided_by_3": predict_knn(scale_rows, (0, 0), 1, (1, 3))},
    }



## 固定例を予想して実行

入力(2,3)：近傍はT4・T2・T3、距離の二乗は1・5・9。k=1はB、k=3はA。木はx≤4でA／B、入力の予測はA。投票B・A・Aの集約はA。

In [ ]:
baseline_result = run_experiment()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))

## 成績の読み方

k=1は学習7/7・検証2/4、k=3は学習6/7・検証4/4。検証でk=3を選び、選択後にテスト4/4を確認する。固定例の結果で、一般的な最適kや実物の性能を示さない。投票は仕組みの例であり、投票構成を検証で選んだ性能比較ではない。

この後は一度に一条件だけ変更する。固定例の`baseline_result`は変更しない。テストを見て変更した場合、同じテストを独立した最終評価と呼ばない。

In [ ]:
changed_query = (3, 3)
for k in (1, 3):
    print(k, [(str(d), i, label) for d, i, label in nearest(TRAIN, changed_query, k)],
          predict_knn(TRAIN, changed_query, k))

## 尺度だけを変更

原点から(0,3,A)と(2,0,B)を比べる。元の距離の二乗は9と4。yを3で割ると1と4となり、k=1の予測はBからAへ。これは指定した変換の例。標準化の尺度を推定する場合は学習側だけで決める。

In [ ]:
scale_rows = (("S1", 0, 3, "A"), ("S2", 2, 0, "B"))
for scales in ((1, 1), (1, 3)):
    print(scales, predict_knn(scale_rows, (0, 0), 1, scales))

## 学習ラベルだけを変更して木を再学習

T4のBをAへ変更した架空の比較例。原データは上書きしない。分岐前はAが3・Bが4、変更後はAが4・Bが3となる。しきい値が同じでも葉の不純度が変わるかを観察する。ラベルを都合よく書き換えて成績を上げる手順ではない。

In [ ]:
changed_train = tuple((i, x, y, "A" if i == "T4" else label)
                      for i, x, y, label in TRAIN)
print("元の木:", fit_stump(TRAIN))
print("比較用の木:", fit_stump(changed_train))

## 振り返り

入力変更による推論と、学習データ変更による再学習を区別できるか。距離順、投票、分岐と不純度から出力の理由を説明する。追加の結果を保存する場合は`programs/outputs/`へ分ける。